## Import thư viện 


In [1]:
import subprocess, sys
subprocess.run(["pip", "install", "pymupdf", "pytesseract", "pillow",
                "transformers", "accelerate", "scikit-learn", "-q"], check=True)
subprocess.run(["apt-get", "install", "-y", "-q",
                "tesseract-ocr", "tesseract-ocr-vie"], check=True)
import fitz                         # PyMuPDF
import pytesseract
from PIL import Image
import io
import re
import numpy as np
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 69.7 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  tesseract-ocr-vie
0 upgraded, 1 newly installed, 0 to remove and 133 not upgraded.
Need to get 417 kB of archives.
After this operation, 546 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-vie all 1:4.00~git30-7274cfa-1.1 [417 kB]
Fetched 417 kB in 0s (1,243 kB/s)
Selecting previously unselected package tesseract-ocr-vie.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-vie_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-vie (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-vie (1:4.00~git30-7274cfa-1.1) ...


## **AI Summarize System is starting** 

##  Các hàm để chạy Pipe line

In [2]:
# ==============================================================================
# DataOps Pipeline: PDF → Tóm tắt với Qwen 2.5-3B
# Môi trường: Kaggle Notebook (GPU T4)
# ==============================================================================

# ======================== CÀI ĐẶT THƯ VIỆN ========================
# Chạy cell này trước trong Kaggle:
# !pip install pymupdf pytesseract pillow transformers accelerate -q
# !apt-get install tesseract-ocr tesseract-ocr-vie -q  ← nếu PDF tiếng Việt

# ==============================================================================
# ⚙️ CẤU HÌNH CHUNG
# ==============================================================================
CONFIG = {
    "pdf_path": "/kaggle/input/datasets/vietnguyencong123/file-pdf-for-dataops/1.pdf",

    # B1 - Ngưỡng phân loại: nếu tỉ lệ text_chars/total_area < ngưỡng → nhóm A (cần OCR)
    "ocr_threshold": 0.05,          # Có thể điều chỉnh tuỳ PDF

    # B3 - Chunking
    "context_limit": 8192,           # Tăng từ 2048 lên 8192 để AI đọc được nhiều hơn
    "max_tokens_per_chunk": 150,
    "k_ratio": 0.8,                # Hệ số bù trừ: dùng 60% context cho chunks
    # → K = int((context_limit / max_tokens_per_chunk) * k_ratio)
    # → K = int((2048 / 150) * 0.6) ≈ 8 chunks

    # B5 - Model
    "model_name": "Qwen/Qwen2.5-3B-Instruct",
    "max_new_tokens": 512,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    # OCR language (vie+eng hoặc eng)
    "ocr_lang": "vie+eng",
}


# ==============================================================================
# BƯỚC 1: PHÂN LOẠI TRANG (Heuristic — tỉ lệ text/ảnh)
# ==============================================================================
def classify_pages(pdf_path: str, threshold: float) -> dict:
    """
    Trả về dict: { page_index: "A" hoặc "B" }
    - Nhóm A: trang ít text selectable → cần OCR
    - Nhóm B: trang có đủ text selectable → không cần OCR

    Heuristic:
        score = len(text_on_page) / (page_width * page_height)
        score < threshold → Nhóm A
    """
    doc = fitz.open(pdf_path)
    classification = {}

    for i, page in enumerate(doc):
        text = page.get_text("text").strip()
        rect = page.rect
        page_area = rect.width * rect.height

        # Tránh chia 0
        if page_area == 0:
            classification[i] = "A"
            continue

        score = len(text) / page_area
        classification[i] = "B" if score >= threshold else "A"
        print(f"  Trang {i+1}: score={score:.4f} → Nhóm {classification[i]}")

    doc.close()
    return classification


# ==============================================================================
# BƯỚC 2: TRÍCH XUẤT VĂN BẢN
# ==============================================================================
def extract_text_from_page_ocr(page, lang: str) -> str:
    """Render trang thành ảnh rồi dùng Tesseract OCR."""
    mat = fitz.Matrix(2.0, 2.0)            # Scale 2x để OCR chính xác hơn
    pix = page.get_pixmap(matrix=mat)
    img_bytes = pix.tobytes("png")
    img = Image.open(io.BytesIO(img_bytes))
    text = pytesseract.image_to_string(img, lang=lang)
    return text.strip()


def extract_text_from_page_native(page) -> str:
    """Trích xuất text trực tiếp từ PDF (không cần OCR)."""
    return page.get_text("text").strip()


def extract_all_text(pdf_path: str, classification: dict, ocr_lang: str) -> list:
    """
    Trả về list of dict:
    [{ "page": int, "group": "A"/"B", "text": str }, ...]
    """
    doc = fitz.open(pdf_path)
    results = []

    for i, page in enumerate(doc):
        group = classification.get(i, "B")
        if group == "A":
            text = extract_text_from_page_ocr(page, ocr_lang)
            print(f"  Trang {i+1} [Nhóm A - OCR]: {len(text)} ký tự")
        else:
            text = extract_text_from_page_native(page)
            print(f"  Trang {i+1} [Nhóm B - Native]: {len(text)} ký tự")

        results.append({"page": i + 1, "group": group, "text": text})

    doc.close()
    return results


# ==============================================================================
# BƯỚC 3: CHUNKING (≤ 150 tokens, theo câu)
# ==============================================================================
def simple_tokenize_count(text: str) -> int:
    """Ước tính số tokens bằng cách tách từ (nhanh, không cần model tokenizer)."""
    return len(text.split())


def split_into_sentences(text: str) -> list:
    """Tách văn bản thành câu dựa trên dấu câu."""
    # Hỗ trợ cả tiếng Việt và tiếng Anh
    sentences = re.split(r'(?<=[.!?।])\s+|(?<=\n)\s*', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences


def chunk_text(pages_data: list, max_tokens: int) -> list:
    """
    Gộp text từ tất cả trang → tách câu → tạo chunks ≤ max_tokens.
    Chiến lược: greedy — thêm câu vào chunk hiện tại cho đến khi đầy.

    Trả về: list of str (mỗi phần tử là 1 chunk)
    """
    # Gộp toàn bộ text theo thứ tự trang
    full_text = "\n".join(
        p["text"] for p in pages_data if p["text"]
    )

    sentences = split_into_sentences(full_text)
    chunks = []
    current_chunk = []
    current_count = 0

    for sent in sentences:
        sent_tokens = simple_tokenize_count(sent)

        # Nếu 1 câu đơn lẻ đã vượt max_tokens → cắt câu đó
        if sent_tokens > max_tokens:
            words = sent.split()
            sub_chunk = []
            sub_count = 0
            for word in words:
                if sub_count + 1 > max_tokens:
                    chunks.append(" ".join(sub_chunk))
                    sub_chunk = [word]
                    sub_count = 1
                else:
                    sub_chunk.append(word)
                    sub_count += 1
            if sub_chunk:
                # Thêm phần còn lại vào chunk hiện tại nếu còn chỗ
                remaining = " ".join(sub_chunk)
                rem_count = simple_tokenize_count(remaining)
                if current_count + rem_count <= max_tokens:
                    current_chunk.append(remaining)
                    current_count += rem_count
                else:
                    if current_chunk:
                        chunks.append(" ".join(current_chunk))
                    current_chunk = [remaining]
                    current_count = rem_count
            continue

        if current_count + sent_tokens <= max_tokens:
            current_chunk.append(sent)
            current_count += sent_tokens
        else:
            if current_chunk:
                chunks.append(" ".join(current_chunk))
            current_chunk = [sent]
            current_count = sent_tokens

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    print(f"  Tổng số chunks: {len(chunks)}")
    return chunks


# ==============================================================================
# BƯỚC 4: TEXTRANK — Chọn Top K Chunks
# ==============================================================================
def build_tfidf_matrix(chunks: list) -> np.ndarray:
    """
    Xây dựng ma trận TF-IDF thủ công (không dùng sklearn để tránh phụ thuộc).
    Trả về ma trận (n_chunks x n_vocab).
    """
    # Tokenize
    tokenized = [chunk.lower().split() for chunk in chunks]
    vocab = list(set(w for tokens in tokenized for w in tokens))
    word2idx = {w: i for i, w in enumerate(vocab)}

    # TF
    tf_matrix = np.zeros((len(chunks), len(vocab)))
    for i, tokens in enumerate(tokenized):
        for word in tokens:
            tf_matrix[i][word2idx[word]] += 1
        if len(tokens) > 0:
            tf_matrix[i] /= len(tokens)

    # IDF
    df = np.sum(tf_matrix > 0, axis=0)
    idf = np.log((len(chunks) + 1) / (df + 1)) + 1

    return tf_matrix * idf


def textrank(chunks: list, damping: float = 0.85, max_iter: int = 100, tol: float = 1e-4) -> np.ndarray:
    """
    Thuật toán TextRank cho chunks.
    Trả về mảng scores (1 giá trị mỗi chunk).
    """
    if len(chunks) == 0:
        return np.array([])

    # Bước 1: Vector hoá chunks bằng TF-IDF
    tfidf = build_tfidf_matrix(chunks)

    # Bước 2: Ma trận tương đồng cosine giữa các chunks
    sim_matrix = cosine_similarity(tfidf)

    # Bước 3: Chuẩn hoá hàng (row-wise)
    row_sums = sim_matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1                # Tránh chia 0
    sim_matrix = sim_matrix / row_sums

    # Bước 4: Power iteration
    n = len(chunks)
    scores = np.ones(n) / n

    for _ in range(max_iter):
        new_scores = (1 - damping) / n + damping * sim_matrix.T @ scores
        if np.abs(new_scores - scores).sum() < tol:
            break
        scores = new_scores

    return scores


def compute_k(target_word_count, max_tokens_per_chunk, input_limit=8192):
    """
    Muốn output ~N từ → cần input ~N * 3 tokens (hệ số 3x)
    vì model cần đọc nhiều hơn những gì nó viết ra
    """
    tokens_needed = int(target_word_count * 3)           # 1000 từ → cần ~3000 tokens input
    k = tokens_needed // max_tokens_per_chunk            # 3000 / 150 = 20 chunks
    max_k = input_limit // max_tokens_per_chunk          # không vượt context window
    k = min(k, max_k)
    return max(k, 1)


def select_top_k_chunks(chunks: list, scores: np.ndarray, k: int) -> list:
    """
    Chọn top K chunks có điểm TextRank cao nhất.
    Giữ nguyên thứ tự xuất hiện trong tài liệu (không đảo lộn).
    """
    if len(chunks) <= k:
        return chunks

    top_indices = np.argsort(scores)[::-1][:k]
    top_indices_sorted = sorted(top_indices)          # Giữ thứ tự tự nhiên
    selected = [chunks[i] for i in top_indices_sorted]

    print(f"  K = {k} | Đã chọn {len(selected)} chunks (điểm cao nhất)")
    return selected


# ==============================================================================
# BƯỚC 5 & 6: TẢI MODEL QWEN 2.5-3B VÀ TÓM TẮT
# ==============================================================================
def summarize(top_k_chunks, tokenizer, model, target_word_count, device):
    context = "\n\n".join(
        [f"[Đoạn {i+1}]: {chunk}" for i, chunk in enumerate(top_k_chunks)]
    )

    prompt = (
        "Dưới đây là các đoạn thông tin quan trọng nhất được trích xuất từ một tài liệu. "
        "Nhiệm vụ của bạn là xâu chuỗi, kết nối và biên tập thành một bài viết hoàn chỉnh, mạch lạc.\n"
        "Yêu cầu BẮT BUỘC:\n"
        "- Giữ nguyên các ý tưởng và thông tin cốt lõi.\n"
        "- Thêm từ nối, dẫn dắt, mở rộng ý để bài văn trôi chảy và đủ chiều sâu.\n"
        f"- ĐỘ DÀI BẮT BUỘC: viết ĐẦY ĐỦ {target_word_count} từ. "
        f"Nếu chưa đủ {target_word_count} từ thì PHẢI tiếp tục viết thêm.\n"
        "- Viết bằng tiếng Việt.\n\n"
        f"Các đoạn thông tin gốc:\n{context}\n\n"
        "Văn bản hoàn chỉnh:"
    )

    messages = [
        {"role": "system", "content": (
            "Bạn là biên tập viên AI chuyên nghiệp. "
            f"Bạn PHẢI viết đủ {target_word_count} từ, không được dừng sớm hơn."
        )},
        {"role": "user", "content": prompt},
    ]

    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(device)

    # Tiếng Việt ~1.5 tokens/từ, buffer 50% để chắc chắn đủ
    dynamic_max_tokens = int(target_word_count * 1.5 * 1.5)   # 1000 từ → 2250 tokens

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=dynamic_max_tokens,
            do_sample=True,          # Bật sampling để tránh dừng sớm
            temperature=0.7,         # Giữ chất lượng nhưng đa dạng hơn
            top_p=0.9,
            repetition_penalty=1.15, # Giảm từ 1.3 → 1.15, đủ chống lặp mà không dừng sớm
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    raw_text  = tokenizer.decode(generated, skip_special_tokens=True).strip()

    # Bước 1: Loại bỏ đoạn lặp lại (dùng fingerprint dài hơn = ít false positive)
    raw_text = _remove_repetition(raw_text)

    # Bước 2: Cắt chính xác theo số từ target (chỉ cắt nếu VƯỢT QUÁ)
    words = raw_text.split()
    actual = len(words)
    print(f"  [Debug] Số từ raw sau generate: {actual}")

    if actual > int(target_word_count * 1.1):   # chỉ cắt nếu vượt 10%
        truncated = " ".join(words[:target_word_count])
        last_punct = max(
            truncated.rfind("."),
            truncated.rfind("!"),
            truncated.rfind("?"),
        )
        if last_punct > len(truncated) * 0.7:
            raw_text = truncated[:last_punct + 1]
        else:
            raw_text = truncated

    final_count = len(raw_text.split())
    print(f"  [Debug] Số từ sau xử lý: {final_count} / {target_word_count} từ")
    return raw_text.strip()


def _remove_repetition(text: str) -> str:
    """Phát hiện đoạn văn lặp lại bằng fingerprint 80 ký tự (giảm false positive)."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    if len(paragraphs) <= 1:
        return text
    seen, result = [], []
    for para in paragraphs:
        fingerprint = para[:80]   # tăng từ 50 → 80 ký tự
        if fingerprint in seen:
            break
        seen.append(fingerprint)
        result.append(para)
    return "\n\n".join(result)
# ==============================================================================
# 🚀 PIPELINE CHÍNH
# ==============================================================================
def run_pipeline(config: dict, target_word_count: int, tokenizer, model) -> str:
    """
    Tham số:
        config           : dict cấu hình chung
        target_word_count: số từ tóm tắt mong muốn (do người dùng nhập)
        tokenizer        : tokenizer đã tải sẵn từ bên ngoài
        model            : model đã tải sẵn từ bên ngoài
    """
    print("=" * 60)
    print("📄 PIPELINE: PDF → Tóm tắt kết hợp căn chỉnh ngữ nghĩa với Qwen 2.5-3B")
    print(f"   Mục tiêu: {target_word_count} từ")
    print("=" * 60)

    # ── B1: Phân loại trang ──────────────────────────────────────
    print("\n[B1] Phân loại trang PDF ...")
    classification = classify_pages(config["pdf_path"], config["ocr_threshold"])
    n_A = sum(1 for v in classification.values() if v == "A")
    n_B = sum(1 for v in classification.values() if v == "B")
    print(f"  → Nhóm A (OCR): {n_A} trang | Nhóm B (Native): {n_B} trang")

    # ── B2: Trích xuất văn bản ───────────────────────────────────
    print("\n[B2] Trích xuất văn bản ...")
    pages_data = extract_all_text(
        config["pdf_path"], classification, config["ocr_lang"]
    )

    # ── B3: Chunking ─────────────────────────────────────────────
    print("\n[B3] Chunking văn bản (≤ 150 tokens/chunk) ...")
    chunks = chunk_text(pages_data, config["max_tokens_per_chunk"])

    if not chunks:
        return "⚠️ Không trích xuất được văn bản từ PDF."

    # ── B4: TextRank + Chọn Top K ────────────────────────────────
    print("\n[B4] Tính TextRank và chọn Top K chunks ...")
    scores = textrank(chunks)
    k = compute_k(
        target_word_count,                   # ← số từ người dùng nhập
        config["max_tokens_per_chunk"],
        input_limit=config["context_limit"],
    )
    print(f"  Công thức K: int({target_word_count} × 3 / {config['max_tokens_per_chunk']}) = {k} chunks")
    top_k_chunks = select_top_k_chunks(chunks, scores, k)

    # ── B6: Sinh tóm tắt ─────────────────────────────────────────
    print("\n[B6] Sinh tóm tắt ...")
    summary = summarize(
        top_k_chunks,
        tokenizer,
        model,
        target_word_count,                   # ← truyền đúng số từ vào đây
        config["device"],
    )

    print("\n" + "=" * 60)
    print("✅ KẾT QUẢ TÓM TẮT:")
    print("=" * 60)
    return summary



## Download Model Qwen 2.5-3B 


In [3]:
# Chạy cell này ĐÚNG 1 LẦN khi mới mở Kaggle
!pip install pymupdf pytesseract pillow transformers accelerate -q
!apt-get install tesseract-ocr tesseract-ocr-vie -q

import fitz 
import pytesseract
from PIL import Image
import io, re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "Qwen/Qwen2.5-3B-Instruct"

print("Đang tải model vào VRAM...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("✅ Tải model thành công!")

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-vie is already the newest version (1:4.00~git30-7274cfa-1.1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
Đang tải model vào VRAM...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Tải model thành công!


## Run Pipeline

In [12]:
so_tu_nhap = input("👉 Nhập số lượng từ bạn muốn tóm tắt: ")
target_limit = int(so_tu_nhap) if so_tu_nhap.strip().isdigit() else 250
print(f"\n[Hệ thống] Đã nhận giới hạn: {target_limit} từ. Bắt đầu xử lý...\n")

ket_qua = run_pipeline(
    config=CONFIG,
    target_word_count=target_limit,   # ← truyền số từ vào đây
    tokenizer=tokenizer,              # ← dùng tokenizer đã tải sẵn ở cell trước
    model=model,                      # ← dùng model đã tải sẵn ở cell trước
)

from IPython.display import display, Markdown
display(Markdown(ket_qua))

👉 Nhập số lượng từ bạn muốn tóm tắt:  1000



[Hệ thống] Đã nhận giới hạn: 1000 từ. Bắt đầu xử lý...

📄 PIPELINE: PDF → Tóm tắt kết hợp căn chỉnh ngữ nghĩa với Qwen 2.5-3B
   Mục tiêu: 1000 từ

[B1] Phân loại trang PDF ...
  Trang 1: score=0.0058 → Nhóm A
  Trang 2: score=0.0060 → Nhóm A
  Trang 3: score=0.0065 → Nhóm A
  Trang 4: score=0.0067 → Nhóm A
  Trang 5: score=0.0067 → Nhóm A
  Trang 6: score=0.0070 → Nhóm A
  Trang 7: score=0.0070 → Nhóm A
  Trang 8: score=0.0070 → Nhóm A
  Trang 9: score=0.0073 → Nhóm A
  Trang 10: score=0.0070 → Nhóm A
  Trang 11: score=0.0077 → Nhóm A
  Trang 12: score=0.0074 → Nhóm A
  Trang 13: score=0.0067 → Nhóm A
  → Nhóm A (OCR): 13 trang | Nhóm B (Native): 0 trang

[B2] Trích xuất văn bản ...
  Trang 1 [Nhóm A - OCR]: 2629 ký tự
  Trang 2 [Nhóm A - OCR]: 2918 ký tự
  Trang 3 [Nhóm A - OCR]: 3184 ký tự
  Trang 4 [Nhóm A - OCR]: 3267 ký tự
  Trang 5 [Nhóm A - OCR]: 3280 ký tự
  Trang 6 [Nhóm A - OCR]: 3408 ký tự
  Trang 7 [Nhóm A - OCR]: 3424 ký tự
  Trang 8 [Nhóm A - OCR]: 3440 ký tự
  Trang 9 

Chúng ta bắt đầu câu chuyện với một điểm khởi đầu đầy đủ và rõ ràng về lãnh thổ mà nhà Mạc nhận được từ nhà Lê sơ để lại. Lãnh thổ này trải dài từ Bắc sang Nam, với 13 đạo Thừa tuyên và 1 phủ Phụng Thiên, tạo nên bức tranh đa dạng và phức tạp của vùng đất xưa. Sự tồn tại lâu đời của các đạo Thừa tuyên này không chỉ phản ánh truyền thống lịch sử của Việt Nam mà còn thể hiện tầm nhìn chiến lược của nhà Mạc về việc duy trì liên tục hệ thống quyền lực và quản lý đất đai.

Tiếp theo, chúng ta cần nhấn mạnh rằng kể từ thời gian 27 năm sau khi nhà Lê tái lập (từ 1527 đến 1554), nhà Mạc đã gặp phải một loạt thách thức lịch sử đáng kể. Một trong những yếu tố quan trọng ảnh hưởng đến sự thay đổi này là cuộc chiến giữa Nam triều và Bắc triều. Cuộc chiến này kéo dài suốt từ năm 1554, khiến lãnh thổ nhà Mạc phải thu hẹp đáng kể, chỉ còn lại khu vực từ Ninh Bình trở ra Bắc.

Nhưng mặc dù gặp phải những thử thách lớn, nhà Mạc vẫn tiếp tục thực hiện vai trò của mình. Họ không ngừng cải thiện khả năng quản lý đất đai và cư dân. Việc nhà Mạc cử quan lại có chức vụ quan trọng đến các đạo Thừa tuyên không chỉ nhằm đảm bảo an ninh mà còn khẳng định chủ quyền lãnh thổ. Điều này càng được minh chứng bởi việc nhà Mạc cử các quan lại có học vị cao và chức vụ quan trọng như Tiến sĩ Nguyễn Khắc Cần hoặc Đặng Ất đến cai quản các đạo Thừa tuyên.

Ngoài ra, nhà Mạc còn áp dụng các biện pháp quản lý đất đai và cư dân. Ví dụ, tại Thanh - Nghệ - Tín và Thuận - Quảng, nhà Mạc đã phân phối ruộng đất cho dân cư, trong khi ở các vùng đất xa như Thanh Hoa, nhà Mạc đã cử quan lại có trình độ học vấn cao đến cai quản. Đây không chỉ là bước đi quan trọng trong việc củng cố quyền lực mà còn góp phần nâng cao chất lượng quản lý đất đai.

Tuy nhiên, việc nhà Mạc quản lý rộng rãi không có nghĩa là họ có thể kiểm soát hoàn toàn toàn bộ đất đai. Vì lý do lịch sử và thực tế, nhà Mạc đã phải chia sẻ một phần lãnh thổ với nhà Lê. Điều này dẫn đến sự phân chia lãnh thổ nam-north triều, nhưng không giảm bớt tầm quan trọng của nhà Mạc đối với các vùng đất quan trọng như Thanh - Nghệ - Tín và Thuận - Quảng.

Trước hết, việc nhà Mạc cử quan lại có trình độ học vấn cao và chức vụ quan trọng đến các đạo Thừa tuyên, chẳng hạn như Lê Quang Bí hay Phạm Văn Tuấn, không chỉ giúp họ củng cố quyền lực mà còn chứng tỏ tính ưu tú và lòng trung thành của nhân vật này đối với nhà Mạc. Điều này cho thấy nhà Mạc luôn tìm kiếm nguồn lực trí tuệ và kỹ thuật tiên tiến để hỗ trợ trong quá trình quản lý đất đai và dân cư.

Một ví dụ khác nữa là việc nhà Mạc cử quan lại như Nguyễn Khắc Cần, Trần Phỉ, hoặc Đàm Văn Tiết đến các đạo Thừa tuyên khác, bao gồm cả Hải Dương, Thuận Hóa và Sơn Nam, nhằm quản lý ruộng đất và cư dân một cách hiệu quả. Điều này không chỉ giúp duy trì sự ổn định mà còn tạo điều kiện thuận lợi cho việc phát triển kinh tế và xã hội.

Kể từ năm 1554, nhà Mạc đã không chỉ cải thiện quản lý đất đai mà còn phát triển hệ thống quản lý dân cư. Hệ thống đơn vị hành chính từ cấp Trung ương tới cấp cơ sở được thiết lập, với sự can thiệp trực tiếp từ triều đình. Từ tổng, phủ, xã, giáp, đến thôn, mỗi đơn vị đều có vai trò quan trọng trong việc cai quản đất đai và cư dân.

Những đóng góp của nhà Mạc trong việc quản lý đất đai và dân cư đã mang lại kết quả đáng kể. Trước khi nhà Mạc nắm quyền, lãnh thổ mà nhà Lê sơ để lại có ít ruộng đất và dân cư hơn so với những gì nhà Mạc quản lý sau này. Điều này cho thấy sự phát triển mạnh mẽ về mặt nông nghiệp và dân số đã diễn ra nhờ sự quản lý chặt chẽ và sáng suốt của nhà Mạc.

Cuối cùng, việc nhà Mạc đã tiếp tục duy trì và phát triển những thành tựu đã đạt được từ thời kỳ trước, cũng như giải quyết những thách thức mới xuất hiện, cho thấy họ đã làm việc chăm chỉ và kiên trì. Chính nhờ những nỗ lực này mà nhà Mạc đã giữ vững quyền lực và quản lý đất đai một cách hiệu quả trong suốt gần nửa thế kỷ.

Qua tất cả những gì đã diễn ra, chúng ta có thể thấy rằng nhà Mạc đã không chỉ quản lý một phần lãnh thổ rộng lớn mà còn đã củng cố và mở rộng quyền lực của mình. Mặc dù chịu tác động của các biến động lịch sử, họ vẫn duy trì được vị thế của mình và thậm chí còn mở rộng quyền lực. Điều này không chỉ thể hiện sự linh hoạt và thích nghi của nhà Mạc mà còn minh chứng cho tầm nhìn chiến lược và khả năng lãnh đạo của họ trong thời kỳ ấy.

## Làm giao diện 


In [ ]:
import gradio as gr
import traceback

# Khai báo global phòng trường hợp chưa tải model
TOKENIZER = None
MODEL = None

def build_ui():
    with gr.Blocks(title="PDF Summarizer — Qwen 2.5-3B", theme=gr.themes.Soft()) as demo:

        gr.Markdown("""
        # 📄 PDF Summarizer — Qwen 2.5-3B
        **Pipeline:** PDF → Phân loại trang → OCR/Native → Chunking → TextRank → Tóm tắt
        """)

        with gr.Row():

            # ── Cột trái: INPUT ───────────────────────────────────
            with gr.Column(scale=1, min_width=280):
                gr.Markdown("### ⬆️ Đầu vào")

                pdf_input = gr.File(
                    label="Tải lên file PDF",
                    file_types=[".pdf"],
                    type="filepath",
                )

                word_slider = gr.Slider(
                    minimum=100,
                    maximum=2000,
                    value=300,
                    step=50,
                    label="🎯 Số từ tóm tắt mong muốn (100 – 2000)",
                )

                # Chỉ hiển thị, không tương tác (tránh vòng lặp sự kiện)
                word_display = gr.Number(
                    value=300,
                    label="Số từ đã chọn",
                    interactive=False,
                    precision=0,
                )

                # Sync 1 chiều: slider → display
                word_slider.change(
                    fn=lambda x: x,
                    inputs=word_slider,
                    outputs=word_display,
                )

                run_btn = gr.Button("🚀 Bắt đầu xử lý", variant="primary", size="lg")

                gr.Markdown("""
                ---
                **Ghi chú:**
                - Model đã tải sẵn vào VRAM
                - Mỗi lần xử lý ~30–90 giây tuỳ PDF
                - Hỗ trợ PDF tiếng Việt + tiếng Anh
                """)

            # ── Cột phải: OUTPUT ──────────────────────────────────
            with gr.Column(scale=2):
                gr.Markdown("### 📊 Kết quả")

                with gr.Tabs():
                    with gr.Tab("✅ Tóm tắt"):
                        summary_output = gr.Textbox(
                            label="Nội dung tóm tắt",
                            lines=22,
                            show_copy_button=True,
                            placeholder="Kết quả tóm tắt sẽ xuất hiện ở đây...",
                        )
                        word_count_md = gr.Markdown("")

                    with gr.Tab("🪵 Log xử lý"):
                        log_output = gr.Textbox(
                            label="Chi tiết từng bước pipeline",
                            lines=22,
                            placeholder="Log sẽ xuất hiện ở đây...",
                        )

        # ── Xử lý sự kiện nút bấm ────────────────────────────────
        def on_run(pdf_file, target_words):
            global TOKENIZER, MODEL
        
            # Kiểm tra đầu vào trước
            if pdf_file is None:
                yield "❌ Vui lòng tải lên file PDF.", "", ""
                return
        
            target_words = int(target_words)
            if target_words < 50:
                yield "❌ Số từ tối thiểu là 50.", "", ""
                return
        
            if TOKENIZER is None or MODEL is None:
                yield "❌ Model chưa được tải. Hãy chạy cell tải model trước.", "", ""
                return
        
            CONFIG["pdf_path"] = pdf_file
            log_lines = []
        
            def log(msg):
                print(msg)
                log_lines.append(msg)
        
            try:
                # ── B1 ───────────────────────────────────────────────────
                log("=" * 50)
                log(f"PIPELINE  |  Mục tiêu: {target_words} từ")
                log("=" * 50)
                log("\n[B1] Đang phân loại trang PDF ...")
                yield "\n".join(log_lines), "⏳ Đang xử lý B1: Phân loại trang...", ""
        
                classification = classify_pages(CONFIG["pdf_path"], CONFIG["ocr_threshold"])
                n_A = sum(1 for v in classification.values() if v == "A")
                n_B = sum(1 for v in classification.values() if v == "B")
                log(f"  → Nhóm A (OCR): {n_A} trang | Nhóm B (Native): {n_B} trang")
                yield "\n".join(log_lines), "⏳ Đang xử lý B2: Trích xuất văn bản...", ""
        
                # ── B2 ───────────────────────────────────────────────────
                log("\n[B2] Đang trích xuất văn bản ...")
                if n_A > 0:
                    log(f"  ⚠️ Có {n_A} trang cần OCR — bước này mất ~{n_A * 8} giây, vui lòng chờ...")
                yield "\n".join(log_lines), f"⏳ OCR {n_A} trang... (~{n_A * 8} giây)", ""
        
                pages_data = extract_all_text(CONFIG["pdf_path"], classification, CONFIG["ocr_lang"])
                for p in pages_data:
                    log(f"  Trang {p['page']} [Nhóm {p['group']}]: {len(p['text'])} ký tự")
                yield "\n".join(log_lines), "⏳ Đang xử lý B3: Chunking...", ""
        
                # ── B3 ───────────────────────────────────────────────────
                log("\n[B3] Chunking (≤ 150 tokens/chunk) ...")
                chunks = chunk_text(pages_data, CONFIG["max_tokens_per_chunk"])
                log(f"  → Tổng: {len(chunks)} chunks")
        
                if not chunks:
                    yield "\n".join(log_lines), "⚠️ Không trích xuất được văn bản từ PDF.", ""
                    return
        
                yield "\n".join(log_lines), "⏳ Đang xử lý B4: TextRank...", ""
        
                # ── B4 ───────────────────────────────────────────────────
                log("\n[B4] TextRank + chọn Top K ...")
                scores = textrank(chunks)
                k = compute_k(target_words, CONFIG["max_tokens_per_chunk"], CONFIG["context_limit"])
                log(f"  K = int({target_words}×3 / {CONFIG['max_tokens_per_chunk']}) = {k}")
                top_k_chunks = select_top_k_chunks(chunks, scores, k)
                log(f"  → Đã chọn {len(top_k_chunks)} chunks")
                yield "\n".join(log_lines), "⏳ Đang xử lý B5+B6: Sinh tóm tắt với Qwen 2.5-3B...", ""
        
                # ── B5+B6 ─────────────────────────────────────────────────
                log("\n[B5+B6] Sinh tóm tắt với Qwen 2.5-3B ...")
                log(f"  ⚠️ Bước này mất ~30-60 giây, vui lòng chờ...")
                yield "\n".join(log_lines), "⏳ Qwen đang sinh văn bản... (~30-60 giây)", ""
        
                summary = summarize(top_k_chunks, TOKENIZER, MODEL, target_words, CONFIG["device"])
                wc = len(summary.split())
                log(f"  → Hoàn thành! Số từ thực tế: {wc}")
                log("=" * 50)
        
                label = f"📝 **Số từ thực tế: {wc} / {target_words} từ**"
                yield "\n".join(log_lines), summary, label
        
            except Exception:
                err = traceback.format_exc()
                log(f"\n❌ LỖI:\n{err}")
                yield "\n".join(log_lines), f"❌ Lỗi:\n{err}", ""
        
        
        # ── Trong build_ui, sửa run_btn.click thêm streaming ─────────────────────────
        run_btn.click(
            fn=on_run,
            inputs=[pdf_input, word_slider],
            outputs=[log_output, summary_output, word_count_md],
        )

    return demo


# ── Khởi chạy ────────────────────────────────────────────────────────────────
# Gán TOKENIZER và MODEL từ biến đã tải ở cell trước
TOKENIZER = tokenizer
MODEL     = model

demo = build_ui()
demo.launch(
    share=True,
    debug=True,
    show_error=True,
)

/tmp/ipykernel_57/3592982899.py:9: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="PDF Summarizer — Qwen 2.5-3B", theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://2629e37537948b8d67.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


PIPELINE  |  Mục tiêu: 300 từ

[B1] Đang phân loại trang PDF ...
  Trang 1: score=0.0067 → Nhóm A
  Trang 2: score=0.0075 → Nhóm A
  Trang 3: score=0.0068 → Nhóm A
  Trang 4: score=0.0065 → Nhóm A
  Trang 5: score=0.0048 → Nhóm A
  Trang 6: score=0.0077 → Nhóm A
  Trang 7: score=0.0030 → Nhóm A
  → Nhóm A (OCR): 7 trang | Nhóm B (Native): 0 trang

[B2] Đang trích xuất văn bản ...
  ⚠️ Có 7 trang cần OCR — bước này mất ~56 giây, vui lòng chờ...
  Trang 1 [Nhóm A - OCR]: 3281 ký tự
  Trang 2 [Nhóm A - OCR]: 3667 ký tự
  Trang 3 [Nhóm A - OCR]: 3349 ký tự
  Trang 4 [Nhóm A - OCR]: 3194 ký tự
  Trang 5 [Nhóm A - OCR]: 2333 ký tự
  Trang 6 [Nhóm A - OCR]: 3804 ký tự
  Trang 7 [Nhóm A - OCR]: 1466 ký tự
  Trang 1 [Nhóm A]: 3281 ký tự
  Trang 2 [Nhóm A]: 3667 ký tự
  Trang 3 [Nhóm A]: 3349 ký tự
  Trang 4 [Nhóm A]: 3194 ký tự
  Trang 5 [Nhóm A]: 2333 ký tự
  Trang 6 [Nhóm A]: 3804 ký tự
  Trang 7 [Nhóm A]: 1466 ký tự

[B3] Chunking (≤ 150 tokens/chunk) ...
  Tổng số chunks: 30
  → Tổng: 30 ch